# 04 · Spark SQL Analytics
**Part 2.2 of the assignment.**

Two goals:
1. Show **complex SQL** (CTEs, window functions, `NTILE`, conditional aggregation, `PIVOT`, multi-way joins) over the cleaned data.
2. Show how Spark SQL can query data that **lives in MongoDB and HBase** – by loading their data into DataFrames, registering them as views and **joining across systems in plain SQL**.

> **Why SQL on Spark?** DataFrames and SQL compile to the *same* optimised plan (Catalyst). SQL is the lingua franca for analysts, so the same logic can be reviewed by people who don't write Python – and the query texts are saved to `outputs/spark_sql_queries.sql` for the report.

**Prerequisite:** notebook 03 has written `data/processed/parquet/`.

In [1]:
import sys; sys.path.append("../src")
from config import *
import pandas as pd
from pyspark.sql import functions as F
import federation as FED

spark = get_spark("04-spark-sql")
PQ = PROCESSED / "parquet"
for name in ["users", "products", "categories", "sessions", "page_views", "cart_items", "transactions", "transaction_items"]:
    spark.read.parquet(str(PQ / name)).createOrReplaceTempView(name)
print("Views:", [t.name for t in spark.catalog.listTables()])
pd.set_option("display.max_columns", 40, "display.width", 220)

QUERIES = {}          # name -> SQL text (written to outputs/spark_sql_queries.sql at the end)
def sql(name, text, show=15, save=True):
    """Register, run, time, save and show one query."""
    QUERIES[name] = text.strip()
    with timer(name):
        df = spark.sql(text).toPandas()
    if save: df.to_csv(TAB / f"sql_{name}.csv", index=False)
    display(df.head(show)); return df

Views: ['cart_items', 'categories', 'page_views', 'products', 'sessions', 'transaction_items', 'transactions', 'users']


## Part A · Analytical queries on the Spark views

### A1 · Top-3 products per category by net revenue *(CTE + `RANK()` window function)*
`RANK() OVER (PARTITION BY category …)` ranks inside each category without a self-join.

In [2]:
a1 = sql("top_products_per_category", """
WITH prod_rev AS (
    SELECT product_id, product_name, category_id, category_name,
           SUM(item_net_revenue) AS net_revenue, SUM(quantity) AS units
    FROM transaction_items
    GROUP BY product_id, product_name, category_id, category_name
), ranked AS (
    SELECT *, RANK() OVER (PARTITION BY category_id ORDER BY net_revenue DESC) AS rnk
    FROM prod_rev
)
SELECT category_name, rnk, product_id, product_name, ROUND(net_revenue, 2) AS net_revenue, units
FROM ranked WHERE rnk <= 3
ORDER BY category_name, rnk
""")

[timer] top_products_per_category: 7.64s


,category_name,rnk,product_id,product_name,net_revenue,units
0,"Anderson, Walls and Duncan",1,prod_02419,Team-Oriented Client-Server Monitoring,31386.57,58
1,"Anderson, Walls and Duncan",2,prod_00621,Ameliorated Global Project,30959.55,68
2,"Anderson, Walls and Duncan",3,prod_01984,Customer-Focused Zero Tolerance Pricing Structure,29945.51,53
3,Baldwin Ltd,1,prod_01751,Diverse Motivating Time-Frame,32471.24,102
4,Baldwin Ltd,2,prod_00992,Polarized Optimal Orchestration,29633.73,77
5,Baldwin Ltd,3,prod_01935,Implemented Non-Volatile Application,28670.28,69
6,"Brewer, Bryant and Hickman",1,prod_02442,Profit-Focused Executive Focus Group,31661.24,76
7,"Brewer, Bryant and Hickman",2,prod_01416,Seamless Analyzing Benchmark,28585.35,50
8,"Brewer, Bryant and Hickman",3,prod_03639,Horizontal Executive Open System,27173.01,56
9,Burns-Rodriguez,1,prod_01645,Polarized Static Function,28800.41,69


### A2 · Daily revenue, 7-day moving average and week-over-week growth *(window frames + `LAG`)*
`ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` = a sliding 7-day window; `LAG(revenue, 7)` looks back one week. The first/last day of the data are partial days – interpret the edges with care.

In [3]:
a2 = sql("daily_revenue_ma", """
WITH daily AS (
    SELECT txn_date, COUNT(*) AS orders, SUM(total) AS revenue
    FROM transactions WHERE is_valid
    GROUP BY txn_date
)
SELECT txn_date, orders, ROUND(revenue, 2) AS revenue,
       ROUND(AVG(revenue) OVER (ORDER BY txn_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS revenue_ma7,
       ROUND((revenue - LAG(revenue, 7) OVER (ORDER BY txn_date)) / LAG(revenue, 7) OVER (ORDER BY txn_date) * 100, 1) AS wow_growth_pct
FROM daily ORDER BY txn_date
""", show=10)

[timer] daily_revenue_ma: 3.01s


,txn_date,orders,revenue,revenue_ma7,wow_growth_pct
0,2026-08-11,285,308844.68,308844.68,NaN
1,2026-08-12,567,592528.49,450686.58,NaN
2,2026-08-13,491,515567.33,472313.50,NaN
3,2026-08-14,557,621752.23,509673.18,NaN
4,2026-08-15,543,600055.64,527749.67,NaN
5,2026-08-16,550,570154.46,534817.14,NaN
6,2026-08-17,549,600435.15,544191.14,NaN
7,2026-08-18,536,560130.70,580089.14,81.4
8,2026-08-19,570,634802.63,586128.31,7.1
9,2026-08-20,541,553473.82,591543.52,7.4


### A3 · RFM customer scoring *(`NTILE` quintiles + CASE segmentation)*
**R**ecency (days since last order, relative to the newest order in the data), **F**requency (orders), **M**onetary (total spend). Each dimension is cut into quintiles with `NTILE(5)`; the scores are combined into named segments.

In [4]:
a3 = sql("rfm", """
WITH base AS (
    SELECT user_id, MAX(txn_date) AS last_order, COUNT(*) AS frequency, SUM(total) AS monetary
    FROM transactions WHERE is_valid GROUP BY user_id
), ref AS (SELECT MAX(txn_date) AS ref_date FROM transactions),
rfm AS (
    SELECT b.user_id, DATEDIFF(r.ref_date, b.last_order) AS recency_days, b.frequency, ROUND(b.monetary, 2) AS monetary
    FROM base b CROSS JOIN ref r
), scored AS (
    SELECT *,
           NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,    -- recent = high score
           NTILE(5) OVER (ORDER BY frequency)         AS f_score,
           NTILE(5) OVER (ORDER BY monetary)          AS m_score
    FROM rfm
)
SELECT *, CASE WHEN r_score >= 4 AND f_score >= 4 THEN 'Champions'
               WHEN f_score >= 4                  THEN 'Loyal'
               WHEN r_score >= 4                  THEN 'Recent'
               WHEN r_score <= 2 AND f_score <= 2 THEN 'At risk'
               ELSE 'Regular' END AS segment
FROM scored
""", show=8)
a3.groupby("segment").agg(users=("user_id", "count"), avg_recency=("recency_days", "mean"), avg_orders=("frequency", "mean"), avg_spend=("monetary", "mean")).round(1)

[timer] rfm: 4.30s


,user_id,recency_days,frequency,monetary,r_score,f_score,m_score,segment
0,user_007446,12,1,14.75,3,1,1,Regular
1,user_001860,87,1,18.40,1,1,1,At risk
2,user_000612,30,1,35.06,1,1,1,At risk
3,user_008242,61,1,46.31,1,1,1,At risk
4,user_005902,10,2,52.48,3,1,1,Regular
5,user_003073,37,1,63.51,1,1,1,At risk
6,user_008314,12,3,65.87,3,2,1,Regular
7,user_005965,63,1,96.42,1,1,1,At risk


,users,avg_recency,avg_orders,avg_spend
segment,,,,
At risk,2391,37.2,2.9,3082.8
Champions,2165,4.0,7.3,7864.7
Loyal,1811,19.6,7.1,7626.5
Recent,1811,4.6,3.8,4157.6
Regular,1764,18.3,4.3,4641.6


### A4 · Funnel by device *(conditional aggregation)*
`SUM(CASE WHEN … THEN 1 END)` counts several conditions in **one pass** over the table. `is_purchased` is the reconciled flag from notebook 03 (a real order exists), **not** the generator's unreliable `conversion_status`.

In [5]:
a4 = sql("funnel_by_device", """
SELECT device_type,
       COUNT(*)                                                    AS sessions,
       SUM(CASE WHEN viewed_product_page THEN 1 ELSE 0 END)        AS viewed_product,
       SUM(CASE WHEN has_cart THEN 1 ELSE 0 END)                   AS added_to_cart,
       SUM(CASE WHEN reached_checkout THEN 1 ELSE 0 END)           AS reached_checkout,
       SUM(CASE WHEN is_purchased THEN 1 ELSE 0 END)               AS purchased,
       ROUND(100.0 * SUM(CASE WHEN is_purchased THEN 1 ELSE 0 END) / COUNT(*), 2) AS purchase_rate_pct,
       ROUND(100.0 * SUM(CASE WHEN conversion_status_raw = 'converted' THEN 1 ELSE 0 END) / COUNT(*), 2) AS raw_converted_pct
FROM sessions GROUP BY device_type ORDER BY sessions DESC
""")

[timer] funnel_by_device: 3.11s


,device_type,sessions,viewed_product,added_to_cart,reached_checkout,purchased,purchase_rate_pct,raw_converted_pct
0,desktop,66873,66014,43458,29044,8416,12.59,20.21
1,tablet,66594,65767,43400,29103,8390,12.60,20.47
2,mobile,66533,65641,43312,28940,8288,12.46,20.23


### A5 · Category performance: views → cart adds → sales *(3 CTEs joined)*
Combines **three different tables** (page views, cart lines, order lines) at category level.

In [6]:
a5 = sql("category_conversion", """
WITH v AS (SELECT category_id, COUNT(*) AS product_views FROM page_views
           WHERE page_type = 'product_detail' AND category_id IS NOT NULL GROUP BY category_id),
c AS (SELECT p.category_id, COUNT(*) AS cart_adds FROM cart_items ci JOIN products p ON ci.product_id = p.product_id GROUP BY p.category_id),
s AS (SELECT category_id, SUM(quantity) AS units_sold, SUM(item_net_revenue) AS net_revenue, SUM(est_gross_profit) AS est_gross_profit
      FROM transaction_items GROUP BY category_id)
SELECT cat.category_name, v.product_views, c.cart_adds, s.units_sold,
       ROUND(100.0 * c.cart_adds / v.product_views, 2) AS view_to_cart_pct,
       ROUND(s.net_revenue, 0) AS net_revenue, ROUND(s.est_gross_profit, 0) AS est_gross_profit
FROM categories cat JOIN v ON cat.category_id = v.category_id JOIN c ON cat.category_id = c.category_id JOIN s ON cat.category_id = s.category_id
ORDER BY net_revenue DESC
""")

[timer] category_conversion: 8.63s


,category_name,product_views,cart_adds,units_sold,view_to_cart_pct,net_revenue,est_gross_profit
0,Kelly-Santiago,32297,9652,10341,29.89,2597128.0,535009.0
1,Cameron-Parsons,29457,8953,9401,30.39,2427288.0,667506.0
2,"Anderson, Walls and Duncan",30033,9013,9689,30.01,2425305.0,594200.0
3,"Patterson, Smith and Jones",27586,8402,9061,30.46,2316601.0,625483.0
4,"Ellis, Baker and Wright",27847,8414,9135,30.22,2294903.0,491109.0
5,Hancock Inc,30552,9353,9854,30.61,2291629.0,664573.0
6,Mitchell-Kim,27800,8289,8948,29.82,2286414.0,497296.0
7,Carney-Santos,29721,8987,9441,30.24,2269472.0,590063.0
8,Phillips Inc,29353,8733,9358,29.75,2263469.0,561342.0
9,Bush Inc,28253,8492,9106,30.06,2223591.0,589251.0


### A6 · Views vs. sales per product *(feeds the "top-selling vs most-viewed" chart)*
`LEFT JOIN` from `products` so products with views but no sales (and vice-versa) are kept.

In [7]:
a6 = sql("product_views_vs_sales", """
WITH v AS (SELECT product_id, COUNT(*) AS views FROM page_views WHERE page_type = 'product_detail' AND product_id IS NOT NULL GROUP BY product_id),
     s AS (SELECT product_id, SUM(quantity) AS units_sold, SUM(item_net_revenue) AS net_revenue FROM transaction_items GROUP BY product_id)
SELECT p.product_id, p.product_name, p.category_name, p.current_price,
       COALESCE(v.views, 0) AS views, COALESCE(s.units_sold, 0) AS units_sold, ROUND(COALESCE(s.net_revenue, 0), 2) AS net_revenue
FROM products p LEFT JOIN v ON p.product_id = v.product_id LEFT JOIN s ON p.product_id = s.product_id
ORDER BY views DESC
""", show=8)
print("Correlation between views and units sold:", round(a6[["views", "units_sold"]].corr().iloc[0, 1], 3))

[timer] product_views_vs_sales: 4.20s


,product_id,product_name,category_name,current_price,views,units_sold,net_revenue
0,prod_04857,Integrated Clear-Thinking Capability,"Patterson, Smith and Jones",57.88,193,60,3426.50
1,prod_02991,Intuitive Multimedia Structure,"Ellis, Baker and Wright",211.46,192,46,9557.99
2,prod_02412,Expanded System-Worthy Attitude,Cameron-Parsons,100.30,190,52,5055.13
3,prod_01481,Front-Line Context-Sensitive Protocol,Preston Ltd,135.87,186,44,5726.93
4,prod_01973,Team-Oriented Value-Added Array,James LLC,89.51,186,60,5227.40
5,prod_00199,Balanced 5Thgeneration Framework,Preston Ltd,68.26,185,40,2675.79
6,prod_02862,Adaptive Mobile Process Improvement,Bush Inc,181.34,185,49,8613.66
7,prod_03323,Triple-Buffered Mobile System Engine,"Anderson, Walls and Duncan",477.83,185,62,28478.67


Correlation between views and units sold: 0.696


### A7 · Traffic & revenue by hour × weekday *(two aggregations, one join key)*

In [8]:
a7 = sql("hour_dow", """
WITH t AS (SELECT dow_num, day_of_week, hour, COUNT(*) AS sessions FROM sessions GROUP BY dow_num, day_of_week, hour),
     r AS (SELECT dayofweek(txn_date) AS dow_num, hour(txn_ts) AS hour, COUNT(*) AS orders, SUM(total) AS revenue
           FROM transactions WHERE is_valid GROUP BY dayofweek(txn_date), hour(txn_ts))
SELECT t.dow_num, t.day_of_week, t.hour, t.sessions, COALESCE(r.orders, 0) AS orders, ROUND(COALESCE(r.revenue, 0), 2) AS revenue
FROM t LEFT JOIN r ON t.dow_num = r.dow_num AND t.hour = r.hour
ORDER BY t.dow_num, t.hour
""", show=5)

[timer] hour_dow: 3.39s


,dow_num,day_of_week,hour,sessions,orders,revenue
0,1,Sun,0,1207,308,322242.11
1,1,Sun,1,1295,319,349896.95
2,1,Sun,2,1150,325,336366.47
3,1,Sun,3,1184,321,349741.24
4,1,Sun,4,1246,308,330235.17


### A8 · Revenue by payment method × order status *(`PIVOT`)*
`PIVOT` turns the distinct values of a column into columns – a classic reporting shape.

In [9]:
a8 = sql("payment_status_pivot", """
SELECT * FROM (SELECT payment_method, status, total FROM transactions WHERE is_valid)
PIVOT (ROUND(SUM(total), 0) FOR status IN ('completed', 'processing', 'shipped', 'delivered'))
ORDER BY payment_method
""")

[timer] payment_status_pivot: 1.53s


,payment_method,completed,processing,shipped,delivered
0,apple_pay,4877946.0,NaN,NaN,NaN
1,bank_transfer,2121968.0,2144250.0,2117282.0,2256074.0
2,credit_card,7020591.0,2182035.0,2233523.0,2228938.0
3,crypto,4839906.0,NaN,NaN,NaN
4,gift_card,2114359.0,2088585.0,2151368.0,2109379.0
5,paypal,7051941.0,2172607.0,2169965.0,2046259.0


> **Observation for the report:** `apple_pay` and `crypto` only ever appear as *completed*, while `bank_transfer` and `gift_card` appear in every status. This is a **generator artefact** – orders linked to a session always use one payment set and the status `completed`; unlinked orders use another set with random statuses. Payment/status patterns in this dataset are therefore synthetic and must not be interpreted as customer behaviour.

### A9 · Top countries by revenue with average order value *(join users ↔ orders)*

In [10]:
a9 = sql("country_revenue", """
SELECT u.country, COUNT(DISTINCT u.user_id) AS buyers, COUNT(*) AS orders,
       ROUND(SUM(t.total), 0) AS revenue, ROUND(AVG(t.total), 2) AS avg_order_value
FROM transactions t JOIN users u ON t.user_id = u.user_id
WHERE t.is_valid GROUP BY u.country ORDER BY revenue DESC LIMIT 15
""")

[timer] country_revenue: 2.59s


,country,buyers,orders,revenue,avg_order_value
0,CH,67,341,384368.0,1127.18
1,SE,64,340,381151.0,1121.03
2,TH,66,342,372440.0,1089.01
3,RW,69,344,370566.0,1077.23
4,ZW,65,347,369742.0,1065.54
5,IE,63,333,356059.0,1069.25
6,PK,57,328,353308.0,1077.16
7,RS,58,319,347696.0,1089.96
8,SI,58,319,347004.0,1087.79
9,BR,63,301,345824.0,1148.92


## Part B · Spark SQL over data that lives in MongoDB and HBase
In the architecture, **MongoDB** holds the user profiles with their embedded purchase summary and **HBase** holds the session stream. Spark can read both and join them – *something neither store can do for the other*.

Two ways to connect:

| Approach | How | Trade-off |
|---|---|---|
| **A. Spark connectors** | `mongo-spark-connector` (`spark.read.format("mongodb")`), HBase-Spark connector | Distributed reads, predicate pushdown – but version-matching jars (Spark ↔ Scala ↔ connector) is time-consuming |
| **B. Client + `createDataFrame`** *(used here)* | `pymongo` / `happybase` pull a **sample or aggregate**, Spark takes over | Simple and robust; the pull is single-threaded, so only suitable for samples/aggregates |

> *Sketch of approach A (not executed – needs the jars):*
> ```python
> spark = (SparkSession.builder
>     .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0")   # pick the build matching your Spark/Scala version
>     .config("spark.mongodb.read.connection.uri", "mongodb://localhost:27017/ecommerce.users").getOrCreate())
> users = spark.read.format("mongodb").load()          # MongoDB collection -> DataFrame
> ```
> For HBase you would use the `hbase-spark` connector with a catalog JSON mapping row key and `family:qualifier` to DataFrame columns.

**Fallback logic:** if MongoDB/HBase are not running, the loaders in `src/federation.py` build the *same-shaped* DataFrames from the Parquet files and say so (`SIMULATED`). The SQL below is identical either way – and "focus on the SQL query logic itself" is exactly what the assignment asks.

In [11]:
mongo_users, mongo_label = FED.mongo_users_df(spark)
hbase_eng,   hbase_label = FED.hbase_engagement_df(spark)
print("mongo_users     <-", mongo_label, "|", mongo_users.count(), "rows")
print("hbase_engagement<-", hbase_label, "|", hbase_eng.count(), "rows")
mongo_users.createOrReplaceTempView("mongo_users"); hbase_eng.createOrReplaceTempView("hbase_engagement")

mongo_users     <- MongoDB (LIVE: users collection) | 10000 rows
hbase_engagement<- HBase (LIVE: user_sessions scan, meta family) | 10000 rows


### B1 · Does engagement (HBase) predict spend (MongoDB)? *(cross-system join + `NTILE`)*
Users are split into engagement **terciles** by number of sessions; for each tercile we compare purchase behaviour taken from the MongoDB profile.

In [12]:
b1 = sql("xsys_engagement_vs_spend", """
WITH j AS (
    SELECT m.user_id, m.country, m.orders, m.total_spent, h.sessions, h.total_seconds, h.total_page_views,
           NTILE(3) OVER (ORDER BY h.sessions) AS eng_tercile
    FROM mongo_users m JOIN hbase_engagement h ON m.user_id = h.user_id
)
SELECT CASE eng_tercile WHEN 1 THEN '1 - low engagement' WHEN 2 THEN '2 - medium' ELSE '3 - high engagement' END AS engagement,
       COUNT(*) AS users, ROUND(AVG(sessions), 1) AS avg_sessions, ROUND(AVG(total_seconds) / 60, 1) AS avg_minutes_browsing,
       ROUND(AVG(orders), 2) AS avg_orders, ROUND(AVG(total_spent), 2) AS avg_spend,
       ROUND(SUM(CASE WHEN orders > 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_buyers
FROM j GROUP BY eng_tercile ORDER BY eng_tercile
""")
print("Sources ->", mongo_label, "+", hbase_label)

[timer] xsys_engagement_vs_spend: 23.03s


,engagement,users,avg_sessions,avg_minutes_browsing,avg_orders,avg_spend,pct_buyers
0,1 - low engagement,3334,15.2,458.9,4.41,4933.48,99.1
1,2 - medium,3333,19.9,601.0,4.97,5359.42,99.4
2,3 - high engagement,3333,24.9,754.1,5.62,5885.34,99.8


Sources -> MongoDB (LIVE: users collection) + HBase (LIVE: user_sessions scan, meta family)


### B2 · Customers with a purchase profile but *no recent browsing* (anti-join across systems)
"Buyers who stopped visiting" – a re-engagement list. `LEFT ANTI JOIN`-style logic through `NOT EXISTS`, using the last session date from HBase and the last purchase from MongoDB.

In [13]:
b2 = sql("xsys_lapsing_buyers", """
WITH ref AS (SELECT MAX(last_session_ts) AS now_ts FROM hbase_engagement)
SELECT m.user_id, m.country, m.orders, ROUND(m.total_spent, 2) AS total_spent,
       DATE(m.last_purchase) AS last_purchase, DATE(h.last_session_ts) AS last_session,
       DATEDIFF(r.now_ts, h.last_session_ts) AS days_since_last_visit
FROM mongo_users m JOIN hbase_engagement h ON m.user_id = h.user_id CROSS JOIN ref r
WHERE m.orders >= 3 AND DATEDIFF(r.now_ts, h.last_session_ts) >= 3
ORDER BY m.total_spent DESC
""", show=10)

[timer] xsys_lapsing_buyers: 34.35s


,user_id,country,orders,total_spent,last_purchase,last_session,days_since_last_visit
0,user_007368,SB,11,19230.90,2026-11-04,2026-11-04,5
1,user_009161,AG,11,18453.05,2026-10-31,2026-11-05,4
2,user_004921,CG,13,18406.05,2026-10-23,2026-11-01,8
3,user_009189,BF,12,18031.18,2026-10-26,2026-10-27,13
4,user_000883,VA,12,17247.24,2026-11-05,2026-11-01,8
5,user_001490,GQ,14,16762.99,2026-11-08,2026-11-02,7
6,user_002188,CF,12,16754.15,2026-11-05,2026-10-17,23
7,user_005480,CV,9,16696.80,2026-10-26,2026-11-04,5
8,user_009935,ZW,12,16676.86,2026-10-29,2026-11-04,5
9,user_001153,BB,11,16637.22,2026-11-01,2026-11-02,7


## Save the SQL for the report
All query texts are written to one `.sql` file (the assignment asks for *"Spark SQL queries used"*).

In [14]:
out = ROOT / "outputs" / "spark_sql_queries.sql"
with open(out, "w") as f:
    f.write("-- Spark SQL queries used in notebook 04 (run against Parquet views produced by notebook 03)\n\n")
    for name, text in QUERIES.items():
        f.write(f"-- {name}\n{text};\n\n")
print("Wrote", out, "with", len(QUERIES), "queries")
spark.stop()

Wrote E:\bigdata\auca-ecommerce-analytics\outputs\spark_sql_queries.sql with 11 queries


### ✅ What you should have after this notebook
* `outputs/tables/sql_*.csv` (11 result tables) and `outputs/spark_sql_queries.sql`.
* A clear statement (printed by B1) of whether Part B used **live** MongoDB/HBase or the **simulated** fallback – say which in the report.

➡ Next: `05_integrated_analytics.ipynb`.